In [ ]:
from raman_data import raman_data

# Display the available dataset catalogue
catalogue = raman_data()
print(catalogue)

In [ ]:
# Finding "rruf datasets"
rruff_datasets = [key for key in catalogue if "rruff" in key.lower()]
print(rruff_datasets)

In [ ]:
dataset_key = "rruff_mineral_raw"  # or "rruff_mineral_preprocess"

ds = raman_data(dataset_key)
# Inspect the dataset
X = ds.spectra 
y = ds.targets
wavenumbers = ds.raman_shifts

print("Spectra shape:", X.shape) 
print("Labels shape:", y.shape)
print("Wavenumber axis shape:", wavenumbers.shape)

In [ ]:
import numpy as np
import pandas as pd
#convert to numpy arrays for easier manipulation
X = np.asarray(ds.spectra)
y = np.asarray(ds.targets)
wavenumbers = np.asarray(ds.raman_shifts)

print(f"Number of spectra: {X.shape[0]}")
print(f"Features per spectrum: {X.shape[1]}")
print(f"Number of labels: {y.shape[0]}")
print(f"Number of wavenumbers: {wavenumbers.shape[0]}")

print("\nData types")
print("X:", X.dtype)
print("y:", y.dtype)
print("wavenumbers:", wavenumbers.dtype)

assert X.ndim == 2, "Expected X to be a 2D array"
assert y.ndim == 1, "Expected y to be a 1D array"
assert X.shape[0] == y.shape[0], "Spectra and labels don't match"
assert X.shape[1] == wavenumbers.shape[0], "Wavenumber axis doesn't match X"

In [ ]:
print("Missing values in spectra:", np.isnan(X).sum())
print("Infinite values in spectra:", np.isinf(X).sum())
print("Missing labels:", pd.isna(y).sum())

print("\nIntensity summary")
print("Minimum:", np.nanmin(X))
print("Maximum:", np.nanmax(X))
print("Mean:", np.nanmean(X))
print("Standard deviation:", np.nanstd(X))

print("\nWavenumber range")
print("Minimum:", wavenumbers.min(), "cm⁻¹")
print("Maximum:", wavenumbers.max(), "cm⁻¹")

In [ ]:
label_counts = (
    pd.Series(y, name="mineral")
    .value_counts()
    .rename_axis("mineral")
    .reset_index(name="n_spectra")
)

print("Number of unique classes:", label_counts.shape[0])
display(label_counts.head(20))

print("\nClass-size summary")
display(label_counts["n_spectra"].describe())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

top_n = 20

plt.figure(figsize=(12, 6))
sns.barplot(
    data=label_counts.head(top_n),
    x="n_spectra",
    y="mineral"
)
plt.title(f"Top {top_n} mineral classes by number of spectra")
plt.xlabel("Number of spectra")
plt.ylabel("Mineral")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Check spacing between Raman-shift points: Savitzky-Golay assumes an evenly sampled axis.
spacing = np.diff(wavenumbers)
if not np.allclose(spacing, spacing[0], rtol=1e-4, atol=1e-8):
    print("WARNING: spacing is not uniform; resample to a uniform Raman-shift grid before Savitzky-Golay filtering.")
else:
    print("Grid is uniformly spaced; SG window sizes below are measured in sample points.")

print("First 10 wavenumbers:")
print(wavenumbers[:10])

print("\nWavenumber spacing:")
print("Mean:", np.mean(spacing))
print("Min :", np.min(spacing))
print("Max :", np.max(spacing))
print("Strictly increasing:", np.all(spacing > 0))
print("Strictly decreasing:", np.all(spacing < 0))

In [ ]:
print("First wavenumber:", wavenumbers[0])
print("Last wavenumber :", wavenumbers[-1])

In [ ]:
# Select one example from the first 5 different classes
example_labels = label_counts["mineral"].head(5)

plt.figure(figsize=(12, 6))

for label in example_labels:
    idx = np.where(y == label)[0][0]

    plt.plot(
        wavenumbers,
        X[idx],
        label=label
    )

plt.xlabel("Raman shift (cm⁻¹)")
plt.ylabel("Intensity")
plt.title("Example raw RRUFF spectra")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
"""Smooth high-frequency noise while preserving local peak shape as much as possible.

The window is in sample points, not cm^-1; only use this directly on a uniform grid.
"""

from scipy.signal import savgol_filter

def smooth_spectrum(spectrum, window_length=11, polyorder=3):
    # Savitzky-Golay requires an odd window longer than the polynomial order.
    if window_length % 2 == 0 or window_length <= polyorder:
        raise ValueError("window_length must be odd and greater than polyorder")
    if window_length > len(spectrum):
        raise ValueError("window_length cannot exceed the number of spectral points")
    return savgol_filter(
        spectrum,
        window_length=window_length,
        polyorder=polyorder
    )

In [ ]:
#Savitzky–Golay filtering test
idx = 0

raw = X[idx]
smoothed = smooth_spectrum(raw)

plt.figure(figsize=(12, 6))

plt.plot(
    wavenumbers,
    raw,
    label="Raw",
    alpha=0.5
)

plt.plot(
    wavenumbers,
    smoothed,
    label="Smoothed"
)

plt.xlabel("Raman shift (cm⁻¹)")
plt.ylabel("Intensity")
plt.title(f"Savitzky–Golay smoothing — {y[idx]}")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Baseline correction estimates broad background; inspect the overlay so real peaks are not removed.
from scipy import sparse
from scipy.sparse.linalg import spsolve

def baseline_als(y, lam=1e5, p=0.01, niter=10):
    """
    Asymmetric Least Squares baseline correction.

    y     : spectrum
    lam   : smoothness parameter
    p     : asymmetry parameter
    niter : number of iterations
    """

    # Larger lam makes a smoother baseline; p controls asymmetry (small p fits below peaks).
    if lam <= 0 or not 0 < p < 1 or niter < 1:
        raise ValueError("Require lam > 0, 0 < p < 1, and niter >= 1")
    y = np.asarray(y, dtype=float)
    if y.ndim != 1 or not np.all(np.isfinite(y)):
        raise ValueError("Spectrum must be a finite one-dimensional array")
    L = len(y)
    if L < 3:
        raise ValueError("At least 3 points are required for ALS baseline correction")

    # Second-order difference matrix
    D = sparse.diags(
        [1, -2, 1],
        [0, 1, 2],
        shape=(L - 2, L)
    )

    w = np.ones(L)

    for _ in range(niter):
        W = sparse.spdiags(w, 0, L, L)

        Z = W + lam * D.T @ D

        z = spsolve(Z, w * y)

        # Peaks above the baseline get low weights; lower points anchor the estimated background.
        w = np.where(y > z, p, 1 - p)

    return z

In [ ]:
#baseline correction test
idx = 0

raw = X[idx]

baseline = baseline_als(
    raw,
    lam=1e5,
    p=0.01
)

corrected = raw - baseline

plt.figure(figsize=(12, 6))

plt.plot(
    wavenumbers,
    raw,
    label="Raw"
)

plt.plot(
    wavenumbers,
    baseline,
    label="Estimated baseline"
)

plt.xlabel("Raman shift (cm⁻¹)")
plt.ylabel("Intensity")
plt.title("Baseline estimation")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    wavenumbers,
    corrected
)

plt.xlabel("Raman shift (cm⁻¹)")
plt.ylabel("Corrected intensity")
plt.title("Baseline-corrected spectrum")
plt.tight_layout()
plt.show()

In [ ]:
def snv(spectrum):
    # SNV rescales each spectrum independently; it removes absolute intensity scale.
    spectrum = np.asarray(spectrum, dtype=float)
    scale = np.std(spectrum)
    if not np.all(np.isfinite(spectrum)) or not np.isfinite(scale) or scale <= np.finfo(float).eps:
        raise ValueError("SNV requires a finite, non-constant spectrum")
    return (spectrum - np.mean(spectrum)) / scale

#test normalisation on the baseline-corrected spectrum
normalised = snv(corrected)

plt.figure(figsize=(12, 6))

plt.plot(
    wavenumbers,
    normalised
)

plt.xlabel("Raman shift (cm⁻¹)")
plt.ylabel("SNV intensity")
plt.title("SNV-normalised spectrum")
plt.tight_layout()
plt.show()

In [ ]:
# Combine the inspected steps; these parameter values are hypotheses to compare, not fixed truths.
def preprocess_spectrum(
    spectrum,
    smoothing_window=11,
    smoothing_polyorder=3,
    baseline_lam=1e5,
    baseline_p=0.01
):
    # 1. Smooth high-frequency noise; the window counts adjacent samples.
    smoothed = smooth_spectrum(
        spectrum,
        window_length=smoothing_window,
        polyorder=smoothing_polyorder
    )

    # 2. Estimate and remove broad background while retaining narrow Raman bands.
    baseline = baseline_als(
        smoothed,
        lam=baseline_lam,
        p=baseline_p
    )

    corrected = smoothed - baseline

    # 3. SNV gives each spectrum zero mean and unit variance.
    #    It discards absolute intensity, so compare with area/vector normalization too.
    normalised = snv(corrected)

    return normalised

# These point-wise transforms do not learn dataset-wide statistics.
X_processed = np.array([
    preprocess_spectrum(spectrum)
    for spectrum in X
])

print("Original shape :", X.shape)
print("Processed shape:", X_processed.shape)

In [ ]:
#compare 5 example spectra, raw vs pre-processing
example_labels = label_counts["mineral"].head(5)

plt.figure(figsize=(12, 6))

for label in example_labels:
    idx = np.where(y == label)[0][0]

    plt.plot(
        wavenumbers,
        X_processed[idx],
        label=str(label)
    )

plt.xlabel("Raman shift (cm⁻¹)")
plt.ylabel("Normalised intensity")
plt.title("Comparison of 5 preprocessed Raman spectra")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()